# Automatic Code Commenter (with language detection)

A small tool that detects a snippet's programming language with an LLM, then asks the model
to add idiomatic docstrings and comments **without changing any logic**. Ships with a Gradio UI
that streams the commented code back live.

Pipeline: `detect_language` (strict-JSON language detector) → `language_of` (guard) →
`add_comment_messages` / `add_comments` → Gradio front end.

> **Attribution:** the Gradio UI (`generate_comments` + the `gr.Blocks` layout) and the LLM
> system prompts in this notebook were generated with Claude (Claude Code). The pipeline design
> and the rest of the code are my own.

In [ ]:
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
import os 
from IPython.display import Markdown, display, Code
import json

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
openai = OpenAI()

In [ ]:
model = "gpt-5-mini"

In [ ]:
def add_comment_messages(LANGUAGE: str, CODE: str) -> list:
    messages = [
        { 
            "role":"system", 
            "content":"""
            You are a code documentation agent. 
            You will be told the programming language explicitly — do not try to redetect it. 
            Add clear, helpful comments to the code using that language's native comment/doc-comment syntax (# for Python/Ruby/Shell/YAML; // and /* */ for C/C++/Java/JS/TS/Go/Rust/C#/Swift/Kotlin/PHP; -- for SQL/Haskell/Lua; <!-- --> for HTML/XML/Markdown; idiomatic docstrings like \"\"\"...\"\"\" for Python, JSDoc for JS/TS, /// for Rust, Javadoc for Java). 
            Never alter logic, variable names, formatting, or whitespace — only add comments. 
            Comment file/module purpose, function/class purpose with params/returns/errors, and non-obvious logic (why, not just what). 
            Do not comment every line or restate obvious code. 
            Match any existing comment style already in the file. 
            Return ONLY the fully commented code — no extra prose, explanation, or markdown fences before or after.
            """
        },
        {
            "role": "user",
            "content": f"Language: {LANGUAGE}\n\nCode:\n{CODE}" 
        }
    ]
    return messages

In [ ]:
def detect_language_messages(code: str) -> str:
    messages = [
        {"role":"system", 
         "content":
         """You are a code language detection agent. Identify the programming language of the given code. 
         If multiple languages are embedded (e.g. JS inside HTML), list the primary language and embedded ones separately. 
         Respond with STRICT JSON only — no prose, no markdown fences, nothing outside the JSON object. 
         Format: {\"language\": \"Python\", \"confidence\": \"high\", \"reason\": \"one sentence citing concrete syntax evidence\", \"embedded_languages\": [], \"file_extension_guess\": \".py\"}. Use \"unknown\" 
         for language if you cannot determine it, and explain why in reason."""
         },
        {"role": "user", "content": f"{code}"}
    ]
    return messages

In [ ]:
# Detector
def detect_language(code: str) -> str:
    response = openai.chat.completions.create(model=model, messages=detect_language_messages(code))
    content = response.choices[0].message.content 
    return json.loads(content)

In [ ]:
def language_of(code: str) -> str | None:
    """Return the detected language name, or None if the model couldn't determine it."""
    language = detect_language(code).get("language", "unknown")
    return None if language.lower() == "unknown" else language

In [ ]:
# Commenter
def add_comments(code: str) -> str: 
    code_lang = language_of(code)
    if code_lang is None:
        raise ValueError("Code language can't be detected")
    response = openai.chat.completions.create(model=model, messages=add_comment_messages(code_lang,code))
    commented = response.choices[0].message.content
    display(Code(commented, language=code_lang.lower()))
    return commented    

In [ ]:
add_comments("print('Hello')")

In [ ]:
# gradio frontend
SAMPLE_CODE = """def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= (1 / j)
        j = i * param1 + param2
        result += (1 / j)
    return result

print(calculate(1000, 4, 1) * 4)
"""


def generate_comments(code):
    """Gradio handler: detect the language, then stream the commented code back.

    Yields (commented_code, status_markdown) tuples so the UI updates live.
    """
    if not code.strip():
        yield "", "⚠️ Paste some code first."
        return

    language = language_of(code)
    if language is None:
        yield "", "⚠️ Could not detect the programming language."
        return

    stream = openai.chat.completions.create(
        model=model,
        messages=add_comment_messages(language, code),
        stream=True,
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result.replace("```", ""), f"✅ Detected language: **{language}**"


with gr.Blocks(title="Auto Code Commenter", theme=gr.themes.Soft()) as ui:
    gr.Markdown(
        "# 📝 Automatic Code Commenter\n"
        "Paste code and get it back with docstrings and comments added — logic left untouched."
    )
    with gr.Row():
        code_in = gr.Code(label="Your code", language="python", value=SAMPLE_CODE)
        code_out = gr.Code(label="Commented code", language="python")
    with gr.Row():
        generate = gr.Button("Generate comments", variant="primary")
        clear = gr.Button("Clear")
    status = gr.Markdown()

    generate.click(generate_comments, inputs=code_in, outputs=[code_out, status])
    clear.click(lambda: (None, None, ""), outputs=[code_in, code_out, status])

ui.launch(inbrowser=True)